# QuantumCircuit.jl Phase 3A Tunable-Coupler Two-Qubit Dynamics


## Outline

This notebook shows the smallest two-qubit-style mixed system currently supported in Phase 3A.
We use a `TunableTransmon - TunableCoupler - Transmon` chain, drive the tunable transmon locally, and read out how population moves through the coupler into the fixed transmon.

1. Build a `tq1-c1-q2` chain and check its low-lying spectrum.
2. Run an undriven baseline and a driven evolution at one safe operating point.
3. Plot the directly driven response on `tq1`.
4. Plot population transfer across `tq1`, `c1`, and `q2`.
5. Compare transferred response at two coupler-flux values.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using UnicodePlots


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Build the mixed system and inspect the static operating point

The coupler flux stays static in Phase 3A. We first confirm that the chain builds cleanly, then sweep the coupler flux to see how the low-lying spectrum shifts before choosing one operating point for the dynamics section.


## Step 1A - Hamiltonian used in this notebook

This notebook uses the current Duffing-style model for a `TunableTransmon - TunableCoupler - Transmon` chain.
For a tunable Duffing-like subsystem $s \in \{\mathrm{tq1}, \mathrm{c1}\}$ the code uses

$$
E_{J,s}^{\mathrm{eff}}(\Phi) = E_{J,s}^{\max} \sqrt{\cos^2(\pi \Phi) + d_s^2 \sin^2(\pi \Phi)},
$$

$$
\omega_s(\Phi) = \sqrt{8 E_{J,s}^{\mathrm{eff}}(\Phi) E_{C,s}} - E_{C,s}, \qquad \alpha_s = -E_{C,s},
$$

$$
\hat H_s(\Phi) = \omega_s(\Phi) \, \hat n_s + \frac{\alpha_s}{2} \, \hat n_s (\hat n_s - I), \qquad \hat n_s = \hat a_s^\dagger \hat a_s.
$$

The fixed transmon `q2` uses the same Duffing form without flux dependence,

$$
\hat H_{q2} = \omega_{q2} \, \hat n_{q2} + \frac{\alpha_{q2}}{2} \, \hat n_{q2} (\hat n_{q2} - I).
$$

The chain is coupled through the current `CapacitiveCoupling` implementation,

$$
\hat H_{\mathrm{int}} = g_{\mathrm{tq1},\mathrm{c1}} \left( \hat a_{\mathrm{tq1}}^\dagger \hat a_{\mathrm{c1}} + \hat a_{\mathrm{tq1}} \hat a_{\mathrm{c1}}^\dagger \right) + g_{\mathrm{c1},\mathrm{q2}} \left( \hat a_{\mathrm{c1}}^\dagger \hat a_{\mathrm{q2}} + \hat a_{\mathrm{c1}} \hat a_{\mathrm{q2}}^\dagger \right),
$$

and the local drive on `tq1` is

$$
\hat H_d(t) = \Omega \cos(\omega_d t) \left( \hat a_{\mathrm{tq1}} + \hat a_{\mathrm{tq1}}^\dagger \right).
$$

So the driven operating point studied below is

$$
\hat H(t; \Phi) = \hat H_{\mathrm{tq1}}(\Phi) + \hat H_{\mathrm{c1}}(\Phi) + \hat H_{q2} + \hat H_{\mathrm{int}} + \hat H_d(t).
$$

For simplicity, the notebook uses the same static flux value $\Phi$ for both `tq1` and `c1` when comparing operating points.


In [3]:
function build_two_qubit_chain(flux::Float64)
    tq1 = TunableTransmon(:tq1; EJmax = 20.0, EC = 0.25, flux = flux, asymmetry = 0.05, ncut = 3)
    c1 = TunableCoupler(:c1; EJmax = 15.0, EC = 0.30, flux = flux, asymmetry = 0.10, ncut = 3)
    q2 = Transmon(:q2; EJ = 19.0, EC = 0.24, ncut = 3)

    return CompositeSystem(
        tq1,
        c1,
        q2,
        CapacitiveCoupling(:tq1, :c1; g = 0.10),
        CapacitiveCoupling(:c1, :q2; g = 0.10),
    )
end

sys = build_two_qubit_chain(0.0)
spec = spectrum(sys; levels = 5)
flux_sweep = simulate_sweep(sys, SweepSpec(:c1, :flux, [0.0, 0.06, 0.12]; levels = 5))
summary = sweep_summary(flux_sweep)

(; low_lying_energies = spec.energies, coupler_flux_values = summary.values, transition_01 = summary.transition_01)


(low_lying_energies = [0.0, 5.621766681951352, 5.8509679464605355, 6.1016882401414785, 11.021457350815057], coupler_flux_values = [0.0, 0.06, 0.12], transition_01 = [5.621766681951352, 5.581171493294301, 5.44382039564993])

## Step 2 - Run baseline and driven dynamics

We start in the product ground state. The baseline evolution should stay quiet, while the driven run applies an `x` drive only to `tq1`.


In [4]:
function run_two_qubit_chain(sys; Ω = 0.25, ωd = 5.95, tlist = collect(range(0.0, 16.0; length = 241)))
    ψ0 = basis_state(sys; tq1 = 0, c1 = 0, q2 = 0)
    drive = SubsystemDrive(
        :tq1_x_drive,
        :tq1,
        :x,
        (p, t) -> p.Ω * cos(p.ωd * t),
    )

    baseline = evolve(
        sys,
        ψ0,
        tlist;
        observables = [ObservableSpec(:ntq1, :tq1, :n), ObservableSpec(:nq2, :q2, :n)],
    )
    driven = evolve(
        sys,
        ψ0,
        tlist;
        drives = [drive],
        observables = [ObservableSpec(:ntq1, :tq1, :n), ObservableSpec(:nq2, :q2, :n)],
        params = (; Ω, ωd),
    )

    return baseline, driven
end

baseline, driven = run_two_qubit_chain(sys)

tq1_baseline = population_trace(baseline, :tq1, 1)
tq1_population = population_trace(driven, :tq1, 1)
c1_population = population_trace(driven, :c1, 1)
q2_population = population_trace(driven, :q2, 1)
tq1_number = observable_trace(driven, :ntq1)
q2_number = observable_trace(driven, :nq2)

(; max_tq1_population = maximum(tq1_population.values), max_c1_population = maximum(c1_population.values), max_q2_population = maximum(q2_population.values), q2_number_peak = maximum(real.(q2_number.values)))


(max_tq1_population = 0.32949587204022973, max_c1_population = 0.34431418302057115, max_q2_population = 0.1800491519530303, q2_number_peak = 0.19557830708939827)

## Step 3 - Compare the directly driven qubit against baseline

This first plot isolates the local response. The undriven baseline stays flat, while the `tq1` excited-state population grows under the drive.


In [5]:
baseline_plot = lineplot(
    tq1_baseline.times,
    tq1_baseline.values;
    name = "baseline",
    title = "tq1 excited-state population: baseline vs driven",
    xlabel = "time",
    ylabel = "P_tq1(|1>)",
)
lineplot!(baseline_plot, tq1_population.times, tq1_population.values; name = "driven")
baseline_plot


                 tq1 excited-state population: baseline vs driven         
                 ┌────────────────────────────────────────┐         
               1 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ baseline
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ driven  
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⢀⡠⠴⠖⠚⠉⠉⠉⠉⠉⠉⠉⠛⠓⠒⠒⠲⠦⠦⠤⠤⠦⠖⠒⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⣀⡤⠖⠋⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
   P_tq1(|1>)    │⠶⠶⠶⠭⠥⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Step 4 - Plot population transfer through the coupler

Now compare all three local excited-state populations. `tq1` is driven directly, `c1` responds in the middle, and `q2` picks up transferred population through the chain.


In [6]:
transfer_plot = lineplot(
    tq1_population.times,
    tq1_population.values;
    name = "tq1",
    title = "Population transfer in a tq1-c1-q2 chain",
    xlabel = "time",
    ylabel = "P(|1>)",
)
lineplot!(transfer_plot, c1_population.times, c1_population.values; name = "c1")
lineplot!(transfer_plot, q2_population.times, q2_population.values; name = "q2")
transfer_plot


              ⠀Population transfer in a tq1-c1-q2 chain⠀    
              ┌────────────────────────────────────────┐    
          0.4 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ tq1
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ c1 
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣠⣠⣠⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠚⠀⠀⠀⠀⠀⠀⠀⠀│ q2 
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠞⠋⠁⠁⠁⠛⠿⣷⣄⡀⠀⠀⠀⠀⠀⣠⠊⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠃⠀⠀⠀⠀⠀⠀⠀⠀⠙⠻⢧⣆⡄⣠⠞⠁⡀⣀⡖⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢈⡕⠱⠳⠛⠛⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⡎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
   P(|1>)     │⠀⠀⠀⠀⠀⠀⠀⠀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡼⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⠀⠀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠏⠀⠀⠀⠀⠀⠀⠀⠀⠀⡴⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⠀⡠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⠀⣀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠎⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⢀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡜⠁⠀⠀⠀⠀⠀⠀⠀⠀⡰⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⠀⠀⡏⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠃⠀⠀⠀⠀⠀⠀⠀⢀⡴⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
              │⠀⠀⢀⠏⠀⠀⠀⠀⠀⠀⠀⣠⠔⠉⠀⠀⠀⠀⠀⠀⢀⡤⠖⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
            0 │⣀⣤⣋⣀⣀⣀⣀⣠⣤

## Step 5 - Compare transferred response at two coupler-flux values

The coupler flux is static during each run, but changing that operating point changes how much of the drive reaches `q2`.


In [7]:
flux_shifted_sys = build_two_qubit_chain(0.12)
_, driven_flux_shifted = run_two_qubit_chain(flux_shifted_sys)
q2_flux_0 = population_trace(driven, :q2, 1)
q2_flux_012 = population_trace(driven_flux_shifted, :q2, 1)

flux_plot = lineplot(
    q2_flux_0.times,
    q2_flux_0.values;
    name = "flux = 0.00",
    title = "Transferred q2 response at two coupler flux points",
    xlabel = "time",
    ylabel = "P_q2(|1>)",
)
lineplot!(flux_plot, q2_flux_012.times, q2_flux_012.values; name = "flux = 0.12")
flux_plot


                 Transferred q2 response at two coupler flux points            
                 ┌────────────────────────────────────────┐            
             0.2 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ flux = 0.00
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀│ flux = 0.12
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡎⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡼⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣰⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
   P_q2(|1>)     │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡞⠀⠀⢀⡔⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡜⠀⠀⡠⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡞⠀⢀⠞⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠞⢀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⢋⡴⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Interpretation

- `tq1` is the only directly driven subsystem in this example.
- `c1` carries a visible intermediate response, which is why the `q2` trace does not remain flat.
- Changing the static coupler flux changes the transfer efficiency even though the drive API itself is still purely subsystem-local.
- `observable_trace(..., :ntq1)` and `observable_trace(..., :nq2)` are useful as a secondary check that the population plots match the number expectation values.


## Current Limits

- This is still a closed-system example. There is no relaxation or dephasing in Phase 3A.
- The coupler flux is a static operating-point parameter here, not a time-dependent flux pulse.
- The local models are Duffing-style approximations, so this notebook is best treated as an intuition-building workflow rather than a calibrated gate model.


## Exercise

Increase one of the coupling strengths or retune the drive frequency `ωd` and compare how the `q2` trace changes.
The quickest useful comparison is to rerun the last plot with a slightly larger `g` or a slightly detuned drive.


In [ ]:
stronger_chain = CompositeSystem(
    TunableTransmon(:tq1; EJmax = 20.0, EC = 0.25, flux = 0.0, asymmetry = 0.05, ncut = 3),
    TunableCoupler(:c1; EJmax = 15.0, EC = 0.30, flux = 0.0, asymmetry = 0.10, ncut = 3),
    Transmon(:q2; EJ = 19.0, EC = 0.24, ncut = 3),
    CapacitiveCoupling(:tq1, :c1; g = 0.12),
    CapacitiveCoupling(:c1, :q2; g = 0.12),
)

_, stronger_driven = run_two_qubit_chain(stronger_chain)
stronger_q2 = population_trace(stronger_driven, :q2, 1)

exercise_plot = lineplot(
    q2_population.times,
    q2_population.values;
    name = "g = 0.10",
    title = "Exercise: stronger coupling increases transferred response",
    xlabel = "time",
    ylabel = "P_q2(|1>)",
)
lineplot!(exercise_plot, stronger_q2.times, stronger_q2.values; name = "g = 0.12")
exercise_plot
